# BRONZE_SIH

**Origem:** arquivos `RDAL{AA}{MM}.dbc` do SIH/SUS (AIH reduzida de Alagoas), um por mês:
`ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/`

Formato `.dbc` (DBF comprimido do DATASUS), descompactado com `pyreaddbc` e lido em
streaming. São carregadas apenas as 21 colunas usadas pelo DW.

**Idempotência:** a tabela é particionada por ano. Antes de carregar um ano, a partição dele é truncada (`TRUNCATE PARTITION FOR`), então reprocessar nunca duplica dados.

**Falhas:** se algum arquivo não puder ser baixado ou lido, o notebook para com erro e a task do Airflow fica vermelha.

Anos processados: variável de ambiente `ANOS_PIPELINE` (padrão `2023,2024`).

In [ ]:
import sys
sys.path.append("..")
from dw_utils import carregar_arquivos_mensais, consultar

In [ ]:
COLUNAS = ['ANO_CMPT', 'MES_CMPT', 'N_AIH', 'IDENT', 'SEQ_AIH5', 'MUNIC_RES', 'MUNIC_MOV', 'CNES', 'PROC_REA', 'COMPLEX', 'ESPEC', 'CAR_INT', 'DIAG_PRINC', 'SEXO', 'IDADE', 'COD_IDADE', 'DT_INTER', 'DT_SAIDA', 'DIAS_PERM', 'VAL_TOT', 'MORTE']

carregar_arquivos_mensais("BRONZE_SIH", "/dissemin/publicos/SIHSUS/200801_/Dados", "RDAL", COLUNAS)

In [ ]:
consultar("""SELECT ANO_CMPT, MES_CMPT, COUNT(*) AS AIHS, SUM(VAL_TOT) AS VALOR FROM BRONZE_SIH GROUP BY ANO_CMPT, MES_CMPT ORDER BY 1, 2""")